In [1]:
%load_ext autoreload
%autoreload 2
import sys, json, time, gc
from pathlib import Path
import numpy as np
import polars as pl
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# ---- ROOT: đi lên đến thư mục chứa cả src/ và configs/ ----
ROOT = Path.cwd().resolve()
while not ((ROOT / 'src').is_dir() and (ROOT / 'configs').is_dir()):
    if ROOT.parent == ROOT:
        raise FileNotFoundError('Không tìm thấy thư mục gốc chứa src/ và configs/')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.features.prepare import META
try:
    from src.features.prepare import AUDIT_COLS
except ImportError:
    AUDIT_COLS = ['person_1__app__sex_738L']

cfg = load_config()
PROC = ROOT / 'data' / 'processed'
REPORTS = ROOT / 'reports'
MODELS = ROOT / 'models'
REPORTS.mkdir(exist_ok=True); MODELS.mkdir(exist_ok=True)

# ---- Cửa sổ thời gian (R5: Challenger tuần 0–43) ----
SPLIT = cfg['split']
assert list(SPLIT['train']) == [0, 43] and list(SPLIT['valid']) == [44, 51]
FIT_WEEKS   = (0, 39)    # fit
ES_WEEKS    = (40, 43)   # early stopping
VALID_WEEKS = (44, 51)   # đánh giá

# ---- Tham số LightGBM (giữ như notebook 05) ----
SEEDS = [42, 43, 44]
N_JOBS = 11
NUM_BOOST_ROUND = 5000
EARLY_STOPPING = 200
LGB_PARAMS = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 200,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'lambda_l2': 10.0,
    'deterministic': True,
    'force_row_wise': True,
    'num_threads': N_JOBS,
    'verbosity': -1,
}

# ---- Metric ổn định của cuộc thi ----
# stability = mean(gini) + 88.0·min(0, a) − 0.5·std(residuals)
# gini = 2·AUC − 1 theo từng WEEK_NUM; y = a·x + b hồi quy tuyến tính (x = WEEK_NUM, y = gini tuần)
def gini_stability(week, y_true, y_score):
    week, y_true, y_score = map(np.asarray, (week, y_true, y_score))
    weeks = np.unique(week)
    ginis = np.array([2 * roc_auc_score(y_true[week == w], y_score[week == w]) - 1 for w in weeks])
    a, b = np.polyfit(weeks.astype(float), ginis, 1)
    residuals = ginis - (a * weeks + b)
    stability = ginis.mean() + 88.0 * min(0.0, a) - 0.5 * residuals.std()
    return {
        'gini_overall': 2 * roc_auc_score(y_true, y_score) - 1,
        'mean_gini': float(ginis.mean()),
        'slope_a': float(a),
        'std_resid': float(residuals.std()),
        'stability': float(stability),
        'weekly': dict(zip(weeks.tolist(), ginis.round(4).tolist())),
    }

# ---- Nạp model_table: CHỈ tuần ≤ 51 (không chạm calib/OOT) ----
t0 = time.time()
lf = pl.scan_parquet(PROC / 'model_table.parquet')
schema = lf.collect_schema()
NON_FEAT = set(META) | set(AUDIT_COLS)
FEATS = [c for c in schema.names() if c not in NON_FEAT]
CAT_FEATS = [c for c in FEATS if schema[c] == pl.String]

df = lf.filter(pl.col('WEEK_NUM') <= VALID_WEEKS[1]).collect(engine='streaming')

print(f'lightgbm {lgb.__version__} | polars {pl.__version__}')
print(f'Shape: {df.shape} | RAM ~{df.estimated_size("gb"):.2f} GB | {time.time()-t0:.1f}s')
print(f'FEATS: {len(FEATS)} | String: {len(CAT_FEATS)} | non-feature: {sorted(NON_FEAT & set(schema.names()))}')
print('Dtype:', dict(pl.DataFrame({'d': [str(schema[c]) for c in FEATS]})['d'].value_counts().iter_rows()))
print(df.group_by('split').agg(pl.len().alias('n'), pl.col('target').mean().round(4).alias('dr'),
                               pl.col('WEEK_NUM').min().alias('w_min'), pl.col('WEEK_NUM').max().alias('w_max'))
        .sort('w_min'))
assert len(FEATS) == 627, len(FEATS)
assert 'person_1__app__sex_738L' not in FEATS

lightgbm 4.7.0 | polars 2.0.0
Shape: (1065921, 634) | RAM ~2.63 GB | 3.0s
FEATS: 627 | String: 33 | non-feature: ['MONTH', 'WEEK_NUM', 'case_id', 'date_decision', 'person_1__app__sex_738L', 'split', 'target']
Dtype: {'UInt32': 112, 'String': 33, 'Float32': 468, 'Int64': 8, 'Boolean': 6}
shape: (2, 5)
┌───────┬────────┬────────┬───────┬───────┐
│ split ┆ n      ┆ dr     ┆ w_min ┆ w_max │
│ ---   ┆ ---    ┆ ---    ┆ ---   ┆ ---   │
│ cat   ┆ u32    ┆ f64    ┆ i64   ┆ i64   │
╞═══════╪════════╪════════╪═══════╪═══════╡
│ train ┆ 841500 ┆ 0.0298 ┆ 0     ┆ 43    │
│ valid ┆ 224421 ┆ 0.0352 ┆ 44    ┆ 51    │
└───────┴────────┴────────┴───────┴───────┘


In [2]:
t0 = time.time()
in_fit = pl.col('WEEK_NUM').is_between(*FIT_WEEKS)

# ---- Danh mục String chỉ lấy từ tuần 0–39 ----
df_fit_cat = df.filter(in_fit).select(CAT_FEATS)
CAT_MAPS = {c: df_fit_cat[c].drop_nulls().unique().sort().to_list() for c in CAT_FEATS}
del df_fit_cat

# ---- Tỷ lệ giá trị chưa gặp trên Valid (chẩn đoán) ----
val_cat = df.filter(pl.col('WEEK_NUM').is_between(*VALID_WEEKS)).select(CAT_FEATS)
unseen_valid = val_cat.select([
    (pl.col(c).is_not_null() & ~pl.col(c).is_in(CAT_MAPS[c])).mean().alias(c) for c in CAT_FEATS
]).row(0, named=True)
del val_cat

# ---- Biểu thức mã hóa, giữ đúng thứ tự FEATS ----
CAT_SET = set(CAT_FEATS)
ENC_EXPRS = [
    (pl.col(c).replace_strict(CAT_MAPS[c], list(range(len(CAT_MAPS[c]))),
                              default=None, return_dtype=pl.Float32)
     if c in CAT_SET else pl.col(c).cast(pl.Float32)).alias(c)
    for c in FEATS
]

def to_xy(lo, hi):
    part = df.filter(pl.col('WEEK_NUM').is_between(lo, hi))
    X = part.select(ENC_EXPRS).to_numpy()
    assert X.dtype == np.float32 and X.shape[1] == len(FEATS)
    return (X, part['target'].to_numpy().astype(np.int8),
            part['WEEK_NUM'].to_numpy(), part['case_id'].to_numpy())

X_fit, y_fit, w_fit, id_fit = to_xy(*FIT_WEEKS)
X_es,  y_es,  w_es,  id_es  = to_xy(*ES_WEEKS)
X_val, y_val, w_val, id_val = to_xy(*VALID_WEEKS)

del df; gc.collect()

# ---- Lưu bảng ánh xạ ----
with open(REPORTS / 'lgb07_cat_maps.json', 'w', encoding='utf-8') as f:
    json.dump(CAT_MAPS, f, ensure_ascii=False)

# ---- Tóm tắt ----
for name, X, y, w in [('fit', X_fit, y_fit, w_fit), ('es', X_es, y_es, w_es), ('valid', X_val, y_val, w_val)]:
    print(f'{name:5s} {X.shape} | dr {y.mean():.4f} | tuần {w.min()}–{w.max()} | {X.nbytes/1e9:.2f} GB')
card = sorted(((c, len(v)) for c, v in CAT_MAPS.items()), key=lambda t: -t[1])
print('\nSố danh mục (top 10):', card[:10])
print('Cột 0 danh mục:', [c for c, n in card if n == 0])
uv = {c: round(v, 4) for c, v in sorted(unseen_valid.items(), key=lambda t: -t[1]) if v > 0}
print(f'\nCột có giá trị chưa gặp trên Valid ({len(uv)}):', uv)
print(f'\n{time.time()-t0:.1f}s')

fit   (754796, 627) | dr 0.0288 | tuần 0–39 | 1.89 GB
es    (86704, 627) | dr 0.0383 | tuần 40–43 | 0.22 GB
valid (224421, 627) | dr 0.0352 | tuần 44–51 | 0.56 GB

Số danh mục (top 10): [('person_1__app__registaddr_zipcode_184M', 3341), ('person_1__app__contaddr_zipcode_807M', 3338), ('person_1__app__empladdr_zipcode_114M', 3102), ('person_1__app__registaddr_district_1083M', 376), ('person_1__app__contaddr_district_15M', 370), ('previouscontdistrict_112M', 198), ('person_1__app__empladdr_district_926M', 198), ('lastcancelreason_561M', 62), ('lastapprcommoditycat_1041M', 42), ('lastrejectcommoditycat_161M', 41)]
Cột 0 danh mục: []

Cột có giá trị chưa gặp trên Valid (8): {'person_1__app__registaddr_district_1083M': 0.0017, 'person_1__app__contaddr_district_15M': 0.0017, 'person_1__app__empladdr_zipcode_114M': 0.0003, 'person_1__app__contaddr_zipcode_807M': 0.0001, 'person_1__app__registaddr_zipcode_184M': 0.0001, 'lastcancelreason_561M': 0.0, 'lastrejectcommoditycat_161M': 0.0, 'lastapp

In [3]:
dtrain = lgb.Dataset(X_fit, label=y_fit, feature_name=FEATS, categorical_feature=CAT_FEATS,
                     params={'data_random_seed': 42, 'verbosity': -1}, free_raw_data=False)
des = lgb.Dataset(X_es, label=y_es, reference=dtrain, free_raw_data=False)

RESULTS, PVAL, IMP = {}, {}, {}
for s in SEEDS:
    t0 = time.time()
    params = {**LGB_PARAMS, 'bagging_seed': s, 'feature_fraction_seed': s}
    booster = lgb.train(
        params, dtrain, num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[des], valid_names=['es'],
        callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False), lgb.log_evaluation(200)],
    )
    bi = booster.best_iteration
    p_val = booster.predict(X_val, num_iteration=bi)
    p_fit = booster.predict(X_fit, num_iteration=bi)
    m = gini_stability(w_val, y_val, p_val)
    gini_fit = float(2 * roc_auc_score(y_fit, p_fit) - 1)

    RESULTS[s] = {
        'best_iter': int(bi),
        'es_gini': float(2 * booster.best_score['es']['auc'] - 1),
        'gini_fit': gini_fit,
        'gap_fit_valid': gini_fit - m['gini_overall'],
        **m,
        'time_s': round(time.time() - t0, 1),
    }
    PVAL[s] = p_val
    IMP[s] = booster.feature_importance('gain', iteration=bi)

    # ---- Lưu ngay sau mỗi seed ----
    booster.save_model(str(MODELS / f'lgb07_s{s}.txt'), num_iteration=bi)
    np.save(REPORTS / f'lgb07_pval_s{s}.npy', p_val)
    np.save(REPORTS / f'lgb07_imp_gain_s{s}.npy', IMP[s])
    with open(REPORTS / 'lgb07_results.json', 'w', encoding='utf-8') as f:
        json.dump({str(k): v for k, v in RESULTS.items()}, f, indent=2)

    r = RESULTS[s]
    print(f"seed {s} | iter {r['best_iter']} | ES {r['es_gini']:.4f} | fit {r['gini_fit']:.4f} | "
          f"Valid {r['gini_overall']:.4f} | gap {r['gap_fit_valid']:.4f} | "
          f"stab {r['stability']:.4f} (mean {r['mean_gini']:.4f}, a {r['slope_a']:+.5f}, "
          f"std_res {r['std_resid']:.4f}) | {r['time_s']:.0f}s")
    del booster, p_fit; gc.collect()

c:\Users\HOME\Downloads\da2-credit-risk\.venv\Lib\site-packages\lightgbm\basic.py:2515: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


[200]	es's auc: 0.8215
seed 42 | iter 136 | ES 0.6439 | fit 0.8305 | Valid 0.6474 | gap 0.1831 | stab 0.6079 (mean 0.6446, a -0.00025, std_res 0.0290) | 192s
[200]	es's auc: 0.822618
seed 43 | iter 195 | ES 0.6455 | fit 0.8563 | Valid 0.6477 | gap 0.2087 | stab 0.6039 (mean 0.6447, a -0.00030, std_res 0.0290) | 154s
[200]	es's auc: 0.821185
seed 44 | iter 140 | ES 0.6439 | fit 0.8326 | Valid 0.6477 | gap 0.1849 | stab 0.6116 (mean 0.6449, a -0.00022, std_res 0.0277) | 137s


In [4]:
import re

# ======== A. Tổng hợp 3 seed ========
COLS = ['best_iter', 'es_gini', 'gini_fit', 'gini_overall', 'gap_fit_valid',
        'mean_gini', 'slope_a', 'std_resid', 'stability']
res = pl.DataFrame([{'seed': s, **{k: RESULTS[s][k] for k in COLS}} for s in SEEDS])
print(res)
summ = {k: (float(res[k].mean()), float(res[k].std(ddof=1))) for k in COLS}
for k in ['gini_overall', 'stability', 'mean_gini', 'slope_a', 'std_resid', 'gap_fit_valid']:
    print(f'{k:14s} {summ[k][0]:.5f} ± {summ[k][1]:.5f}')

# ---- Trung bình dự đoán 3 seed ----
p_mean = np.mean([PVAL[s] for s in SEEDS], axis=0)
m_ens = gini_stability(w_val, y_val, p_mean)
print(f"\nEnsemble 3 seed | Valid {m_ens['gini_overall']:.4f} | stab {m_ens['stability']:.4f} "
      f"(mean {m_ens['mean_gini']:.4f}, a {m_ens['slope_a']:+.5f}, std_res {m_ens['std_resid']:.4f})")

# ---- Gini theo tuần ----
weekly = pl.DataFrame({'WEEK_NUM': list(m_ens['weekly'].keys()),
                       **{f's{s}': list(RESULTS[s]['weekly'].values()) for s in SEEDS},
                       'ens': list(m_ens['weekly'].values())})
print(weekly)

# ---- Lưu dự đoán Valid theo case_id (điểm xuất phát Challenger-H) ----
pl.DataFrame({'case_id': id_val, 'WEEK_NUM': w_val, 'target': y_val,
              **{f'pred_s{s}': PVAL[s] for s in SEEDS}, 'pred_mean': p_mean}
             ).write_parquet(REPORTS / 'lgb07_valid_preds.parquet')
weekly.write_csv(REPORTS / 'lgb07_weekly_gini.csv')

# ---- Feature importance (gain, trung bình 3 seed, chuẩn hóa) ----
g = np.mean([IMP[s] / IMP[s].sum() for s in SEEDS], axis=0)
imp_df = pl.DataFrame({'feature': FEATS, 'gain_share': g}).sort('gain_share', descending=True)
imp_df.write_csv(REPORTS / 'lgb07_importance_gain.csv')
print(f"\nFeature gain = 0: {(imp_df['gain_share'] == 0).sum()} / {len(FEATS)} | "
      f"top 50 chiếm {imp_df['gain_share'][:50].sum():.3f}")
print(imp_df.head(15))

with open(REPORTS / 'lgb07_summary.json', 'w', encoding='utf-8') as f:
    json.dump({'params': LGB_PARAMS, 'seeds': SEEDS, 'num_boost_round': NUM_BOOST_ROUND,
               'early_stopping': EARLY_STOPPING, 'fit_weeks': FIT_WEEKS, 'es_weeks': ES_WEEKS,
               'valid_weeks': VALID_WEEKS, 'n_feats': len(FEATS),
               'per_seed_mean_std': summ, 'ensemble': m_ens}, f, indent=2)

# ======== B. So với notebook 05 (cửa sổ A, bảng 643 feature) ========
cand = [p for d in (REPORTS, MODELS) for p in d.glob('lgb_window_A*.txt')]
print('\nModel notebook 05:', cand)
if cand:
    path05 = cand[0]
    txt = path05.read_text(encoding='utf-8')
    ptxt = txt[txt.index('parameters:'):txt.index('end of parameters')]
    P05 = dict(re.findall(r'\[(\w+): ([^\]]*)\]', ptxt))
    KEYS = ['learning_rate', 'num_leaves', 'min_data_in_leaf', 'feature_fraction', 'bagging_fraction',
            'bagging_freq', 'lambda_l2', 'max_bin', 'max_cat_threshold', 'cat_smooth',
            'min_data_per_group', 'early_stopping_round', 'num_iterations', 'seed', 'deterministic']
    print({k: P05.get(k) for k in KEYS})
    print('Số cây notebook 05:', txt.count('\nTree='))

    b05 = lgb.Booster(model_file=str(path05))
    F05 = b05.feature_name()
    gain05 = b05.feature_importance('gain'); gain05 = gain05 / gain05.sum()
    dropped = [f for f in F05 if f not in set(FEATS)]
    added = [f for f in FEATS if f not in set(F05)]
    share_dropped = {f: round(float(gain05[F05.index(f)]), 4) for f in dropped}
    print(f'\nFeature 05: {len(F05)} | 07: {len(FEATS)} | bị bỏ: {len(dropped)} | mới: {len(added)}')
    print(f'Gain share của nhóm bị bỏ trong model 05: {sum(share_dropped.values()):.4f}')
    print(dict(sorted(share_dropped.items(), key=lambda t: -t[1])))
    print('Mới:', added)

shape: (3, 10)
┌──────┬───────────┬──────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ seed ┆ best_iter ┆ es_gini  ┆ gini_fit ┆ … ┆ mean_gini ┆ slope_a   ┆ std_resid ┆ stability │
│ ---  ┆ ---       ┆ ---      ┆ ---      ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ i64  ┆ i64       ┆ f64      ┆ f64      ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞══════╪═══════════╪══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 42   ┆ 136       ┆ 0.643895 ┆ 0.830482 ┆ … ┆ 0.644563  ┆ -0.000252 ┆ 0.028959  ┆ 0.607901  │
│ 43   ┆ 195       ┆ 0.645496 ┆ 0.856328 ┆ … ┆ 0.644735  ┆ -0.000299 ┆ 0.029047  ┆ 0.603871  │
│ 44   ┆ 140       ┆ 0.64394  ┆ 0.83258  ┆ … ┆ 0.644899  ┆ -0.000221 ┆ 0.027737  ┆ 0.611619  │
└──────┴───────────┴──────────┴──────────┴───┴───────────┴───────────┴───────────┴───────────┘
gini_overall   0.64756 ± 0.00016
stability      0.60780 ± 0.00388
mean_gini      0.64473 ± 0.00017
slope_a        -0.00026 ± 0.000

In [5]:
def run_lgb(tag, override=None, seeds=SEEDS, base=RESULTS, base_pval=PVAL):
    override = override or {}
    out = {'tag': tag, 'override': override, 'seeds': {}}
    pv = []
    for s in seeds:
        t0 = time.time()
        params = {**LGB_PARAMS, **override, 'bagging_seed': s, 'feature_fraction_seed': s}
        b = lgb.train(params, dtrain, num_boost_round=NUM_BOOST_ROUND,
                      valid_sets=[des], valid_names=['es'],
                      callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False)])
        bi = b.best_iteration
        p = b.predict(X_val, num_iteration=bi)
        m = gini_stability(w_val, y_val, p)
        gfit = float(2 * roc_auc_score(y_fit, b.predict(X_fit, num_iteration=bi)) - 1)
        out['seeds'][s] = {'best_iter': int(bi), 'gini_fit': gfit,
                           'gap_fit_valid': gfit - m['gini_overall'], **m,
                           'time_s': round(time.time() - t0, 1)}
        pv.append(p)
        np.save(REPORTS / f'lgb07_{tag}_pval_s{s}.npy', p)
        b.save_model(str(MODELS / f'lgb07_{tag}_s{s}.txt'), num_iteration=bi)
        r = out['seeds'][s]
        d = r['gini_overall'] - base[s]['gini_overall'] if base and s in base else np.nan
        print(f"[{tag}] seed {s} | iter {r['best_iter']} | fit {gfit:.4f} | Valid {r['gini_overall']:.4f} "
              f"(Δ {d:+.4f}) | gap {r['gap_fit_valid']:.4f} | stab {r['stability']:.4f} | {r['time_s']:.0f}s")
        del b; gc.collect()

    p_ens = np.mean(pv, axis=0)
    out['ensemble'] = gini_stability(w_val, y_val, p_ens)
    for k in ['gini_overall', 'stability', 'gap_fit_valid', 'best_iter']:
        v = np.array([out['seeds'][s][k] for s in seeds], dtype=float)
        out[f'{k}_mean_std'] = (float(v.mean()), float(v.std(ddof=1)))

    # ---- So với baseline: ensemble theo tuần ----
    if base_pval:
        m_b = gini_stability(w_val, y_val, np.mean([base_pval[s] for s in seeds], axis=0))
        dw = {w: out['ensemble']['weekly'][w] - m_b['weekly'][w] for w in m_b['weekly']}
        out['vs_base'] = {'d_valid_ens': out['ensemble']['gini_overall'] - m_b['gini_overall'],
                          'd_stab_ens': out['ensemble']['stability'] - m_b['stability'],
                          'weeks_won': int(sum(v > 0 for v in dw.values())),
                          'weekly_delta': {w: round(v, 4) for w, v in dw.items()}}

    with open(REPORTS / f'lgb07_{tag}.json', 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=2, default=str)

    e = out['ensemble']
    print(f"\n[{tag}] per-seed Valid {out['gini_overall_mean_std'][0]:.5f} ± {out['gini_overall_mean_std'][1]:.5f} | "
          f"stab {out['stability_mean_std'][0]:.5f} ± {out['stability_mean_std'][1]:.5f}")
    print(f"[{tag}] ensemble Valid {e['gini_overall']:.4f} | stab {e['stability']:.4f} "
          f"(mean {e['mean_gini']:.4f}, a {e['slope_a']:+.5f}, std_res {e['std_resid']:.4f})")
    if 'vs_base' in out:
        vb = out['vs_base']
        print(f"[{tag}] so baseline (ensemble): ΔValid {vb['d_valid_ens']:+.4f} | Δstab {vb['d_stab_ens']:+.4f} | "
              f"thắng {vb['weeks_won']}/8 tuần | {vb['weekly_delta']}")
    return out

EXP = {}
EXP['mdpg200'] = run_lgb('mdpg200', {'min_data_per_group': 200})

[mdpg200] seed 42 | iter 158 | fit 0.8404 | Valid 0.6477 (Δ +0.0003) | gap 0.1927 | stab 0.6041 | 131s
[mdpg200] seed 43 | iter 133 | fit 0.8272 | Valid 0.6480 (Δ +0.0004) | gap 0.1791 | stab 0.6310 | 125s
[mdpg200] seed 44 | iter 133 | fit 0.8276 | Valid 0.6477 (Δ +0.0001) | gap 0.1798 | stab 0.6305 | 128s

[mdpg200] per-seed Valid 0.64781 ± 0.00019 | stab 0.62185 ± 0.01538
[mdpg200] ensemble Valid 0.6521 | stab 0.6315 (mean 0.6492, a -0.00004, std_res 0.0284)
[mdpg200] so baseline (ensemble): ΔValid -0.0003 | Δstab +0.0143 | thắng 2/8 tuần | {44: -0.0006, 45: -0.0025, 46: 0.0014, 47: -0.001, 48: -0.0018, 49: 0.0028, 50: 0.0, 51: -0.0009}


In [6]:
HC = [c for c, n in card if n >= 150]
print('HC:', HC)

FIT0, ES0, VAL0 = X_fit, X_es, X_val
DS0 = (dtrain, des)

def set_data(keep_idx=None, cats=CAT_FEATS):
    global dtrain, des, X_fit, X_val
    if keep_idx is None:
        X_fit, X_val, Xe, names = FIT0, VAL0, ES0, FEATS
    else:
        X_fit, X_val, Xe = FIT0[:, keep_idx], VAL0[:, keep_idx], ES0[:, keep_idx]
        names = [FEATS[i] for i in keep_idx]
    cats = [c for c in cats if c in set(names)]
    dtrain = lgb.Dataset(X_fit, label=y_fit, feature_name=names, categorical_feature=cats,
                         params={'data_random_seed': 42, 'verbosity': -1}, free_raw_data=False)
    des = lgb.Dataset(Xe, label=y_es, reference=dtrain, free_raw_data=False)
    print(f'set_data: {len(names)} feature | {len(cats)} categorical')

def restore_data():
    global dtrain, des, X_fit, X_val
    dtrain, des = DS0
    X_fit, X_val = FIT0, VAL0
    gc.collect()

# ---- 1. catreg: dữ liệu gốc, chỉ đổi tham số ----
restore_data()
EXP['catreg'] = run_lgb('catreg', {'cat_smooth': 50, 'cat_l2': 50,
                                   'min_data_per_group': 500, 'max_cat_threshold': 16})

# ---- 2. hc_numeric: 7 cột thành feature số ----
set_data(cats=[c for c in CAT_FEATS if c not in HC])
EXP['hc_numeric'] = run_lgb('hc_numeric')

# ---- 3. hc_drop: bỏ 7 cột ----
keep = [i for i, f in enumerate(FEATS) if f not in set(HC)]
set_data(keep_idx=keep)
EXP['hc_drop'] = run_lgb('hc_drop')
restore_data()

# ---- Bảng tổng hợp ----
rows = [{'tag': 'baseline', 'valid_seed': summ['gini_overall'][0], 'valid_ens': m_ens['gini_overall'],
         'stab_ens': m_ens['stability'], 'gap': summ['gap_fit_valid'][0],
         'iter': summ['best_iter'][0], 'd_valid_ens': 0.0, 'weeks_won': None}]
for t, o in EXP.items():
    rows.append({'tag': t, 'valid_seed': o['gini_overall_mean_std'][0], 'valid_ens': o['ensemble']['gini_overall'],
                 'stab_ens': o['ensemble']['stability'], 'gap': o['gap_fit_valid_mean_std'][0],
                 'iter': o['best_iter_mean_std'][0], 'd_valid_ens': o['vs_base']['d_valid_ens'],
                 'weeks_won': o['vs_base']['weeks_won']})
tab = pl.DataFrame(rows)
tab.write_csv(REPORTS / 'lgb07_ablation_hc.csv')
with pl.Config(tbl_cols=-1, tbl_width_chars=200, float_precision=4):
    print(tab)

HC: ['person_1__app__registaddr_zipcode_184M', 'person_1__app__contaddr_zipcode_807M', 'person_1__app__empladdr_zipcode_114M', 'person_1__app__registaddr_district_1083M', 'person_1__app__contaddr_district_15M', 'previouscontdistrict_112M', 'person_1__app__empladdr_district_926M']
[catreg] seed 42 | iter 271 | fit 0.8230 | Valid 0.6795 (Δ +0.0322) | gap 0.1434 | stab 0.6626 | 176s
[catreg] seed 43 | iter 341 | fit 0.8425 | Valid 0.6799 (Δ +0.0322) | gap 0.1626 | stab 0.6635 | 212s
[catreg] seed 44 | iter 245 | fit 0.8156 | Valid 0.6795 (Δ +0.0318) | gap 0.1361 | stab 0.6631 | 180s

[catreg] per-seed Valid 0.67964 ± 0.00022 | stab 0.66307 ± 0.00048
[catreg] ensemble Valid 0.6833 | stab 0.6668 (mean 0.6802, a +0.00081, std_res 0.0266)
[catreg] so baseline (ensemble): ΔValid +0.0309 | Δstab +0.0497 | thắng 8/8 tuần | {44: 0.0281, 45: 0.0331, 46: 0.0238, 47: 0.0234, 48: 0.0323, 49: 0.0354, 50: 0.0389, 51: 0.0299}
set_data: 627 feature | 26 categorical


c:\Users\HOME\Downloads\da2-credit-risk\.venv\Lib\site-packages\lightgbm\basic.py:2515: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


[hc_numeric] seed 42 | iter 396 | fit 0.8227 | Valid 0.6911 (Δ +0.0437) | gap 0.1316 | stab 0.6745 | 203s
[hc_numeric] seed 43 | iter 469 | fit 0.8374 | Valid 0.6911 (Δ +0.0434) | gap 0.1463 | stab 0.6749 | 239s
[hc_numeric] seed 44 | iter 514 | fit 0.8458 | Valid 0.6932 (Δ +0.0455) | gap 0.1527 | stab 0.6779 | 236s

[hc_numeric] per-seed Valid 0.69176 ± 0.00121 | stab 0.67577 ± 0.00186
[hc_numeric] ensemble Valid 0.6958 | stab 0.6798 (mean 0.6925, a +0.00137, std_res 0.0254)
[hc_numeric] so baseline (ensemble): ΔValid +0.0434 | Δstab +0.0626 | thắng 8/8 tuần | {44: 0.034, 45: 0.0501, 46: 0.0346, 47: 0.0332, 48: 0.0448, 49: 0.053, 50: 0.0523, 51: 0.0417}
set_data: 620 feature | 26 categorical


c:\Users\HOME\Downloads\da2-credit-risk\.venv\Lib\site-packages\lightgbm\basic.py:2515: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


[hc_drop] seed 42 | iter 428 | fit 0.8286 | Valid 0.6920 (Δ +0.0447) | gap 0.1366 | stab 0.6760 | 239s
[hc_drop] seed 43 | iter 445 | fit 0.8314 | Valid 0.6922 (Δ +0.0445) | gap 0.1393 | stab 0.6765 | 218s
[hc_drop] seed 44 | iter 394 | fit 0.8217 | Valid 0.6909 (Δ +0.0433) | gap 0.1307 | stab 0.6754 | 187s

[hc_drop] per-seed Valid 0.69171 ± 0.00067 | stab 0.67598 ± 0.00057
[hc_drop] ensemble Valid 0.6956 | stab 0.6800 (mean 0.6925, a +0.00122, std_res 0.0251)
[hc_drop] so baseline (ensemble): ΔValid +0.0432 | Δstab +0.0628 | thắng 8/8 tuần | {44: 0.0347, 45: 0.0495, 46: 0.035, 47: 0.0325, 48: 0.0465, 49: 0.0529, 50: 0.0524, 51: 0.0401}
shape: (5, 8)
┌────────────┬────────────┬───────────┬──────────┬────────┬──────────┬─────────────┬───────────┐
│ tag        ┆ valid_seed ┆ valid_ens ┆ stab_ens ┆ gap    ┆ iter     ┆ d_valid_ens ┆ weeks_won │
│ ---        ┆ ---        ┆ ---       ┆ ---      ┆ ---    ┆ ---      ┆ ---         ┆ ---       │
│ str        ┆ f64        ┆ f64       ┆ f64      

In [7]:
import warnings
warnings.filterwarnings('ignore', message='Overriding the parameters from Reference Dataset')

KEEP_D = [i for i, f in enumerate(FEATS) if f not in set(HC)]
FEATS_D = [FEATS[i] for i in KEEP_D]
CAT_D = [c for c in CAT_FEATS if c not in set(HC)]
CATREG = {'cat_smooth': 50, 'cat_l2': 50, 'min_data_per_group': 500, 'max_cat_threshold': 16}

# ---- Mốc so sánh mới = hc_drop ----
BASE_D = EXP['hc_drop']['seeds']
PVAL_D = {s: np.load(REPORTS / f'lgb07_hc_drop_pval_s{s}.npy') for s in SEEDS}

def set_mats(Xf, Xe, Xv, names, cats):
    global dtrain, des, X_fit, X_val
    X_fit, X_val = Xf, Xv
    dtrain = lgb.Dataset(Xf, label=y_fit, feature_name=names, categorical_feature=cats,
                         params={'data_random_seed': 42, 'verbosity': -1}, free_raw_data=False)
    des = lgb.Dataset(Xe, label=y_es, reference=dtrain, free_raw_data=False)
    gc.collect()
    print(f'set_mats: {len(names)} feature | {len(cats)} categorical')

# ---- Frequency encoding 7 cột HC (tỷ lệ trong tuần 0–39), tính trước từ mã ----
HC_IDX = [FEATS.index(c) for c in HC]
SHARE = {}
for c, j in zip(HC, HC_IDX):
    v = FIT0[:, j]
    SHARE[c] = np.bincount(v[~np.isnan(v)].astype(np.int64), minlength=len(CAT_MAPS[c])) / FIT0.shape[0]

def freq_cols(Xsrc):
    out = np.full((Xsrc.shape[0], len(HC)), np.nan, dtype=np.float32)
    for k, (c, j) in enumerate(zip(HC, HC_IDX)):
        v = Xsrc[:, j]; m = ~np.isnan(v)
        out[m, k] = SHARE[c][v[m].astype(np.int64)]
    return out

F_fit, F_es, F_val = freq_cols(FIT0), freq_cols(ES0), freq_cols(VAL0)
FREQ_NAMES = [f'{c}__freq' for c in HC]

# ---- Ma trận 620 feature ----
Xf_d, Xe_d, Xv_d = FIT0[:, KEEP_D], ES0[:, KEEP_D], VAL0[:, KEEP_D]

# 1. d_catreg
set_mats(Xf_d, Xe_d, Xv_d, FEATS_D, CAT_D)
EXP['d_catreg'] = run_lgb('d_catreg', CATREG, base=BASE_D, base_pval=PVAL_D)

# 2. d_nocat
set_mats(Xf_d, Xe_d, Xv_d, FEATS_D, [])
EXP['d_nocat'] = run_lgb('d_nocat', base=BASE_D, base_pval=PVAL_D)

# 3. d_freq (giải phóng ma trận 620 trước khi chạy để giảm RAM đỉnh)
Xf_f, Xe_f, Xv_f = np.hstack([Xf_d, F_fit]), np.hstack([Xe_d, F_es]), np.hstack([Xv_d, F_val])
set_mats(Xf_f, Xe_f, Xv_f, FEATS_D + FREQ_NAMES, CAT_D)
del Xf_d, Xe_d, Xv_d; gc.collect()
EXP['d_freq'] = run_lgb('d_freq', base=BASE_D, base_pval=PVAL_D)

# ---- Bảng tổng hợp (so với hc_drop) ----
rows = []
for t in ['hc_drop', 'd_catreg', 'd_nocat', 'd_freq']:
    o = EXP[t]
    rows.append({'tag': t, 'valid_seed': o['gini_overall_mean_std'][0], 'seed_std': o['gini_overall_mean_std'][1],
                 'valid_ens': o['ensemble']['gini_overall'], 'stab_ens': o['ensemble']['stability'],
                 'gap': o['gap_fit_valid_mean_std'][0], 'iter': o['best_iter_mean_std'][0],
                 'd_vs_hcdrop': o['vs_base']['d_valid_ens'] if t != 'hc_drop' else 0.0,
                 'weeks_won': o['vs_base']['weeks_won'] if t != 'hc_drop' else None})
tab7 = pl.DataFrame(rows)
tab7.write_csv(REPORTS / 'lgb07_ablation_cat2.csv')
with pl.Config(tbl_cols=-1, tbl_width_chars=200, float_precision=4):
    print(tab7)

set_mats: 620 feature | 26 categorical
[d_catreg] seed 42 | iter 411 | fit 0.8243 | Valid 0.6934 (Δ +0.0014) | gap 0.1308 | stab 0.6767 | 198s
[d_catreg] seed 43 | iter 446 | fit 0.8315 | Valid 0.6915 (Δ -0.0007) | gap 0.1400 | stab 0.6752 | 200s
[d_catreg] seed 44 | iter 515 | fit 0.8445 | Valid 0.6918 (Δ +0.0009) | gap 0.1527 | stab 0.6757 | 222s

[d_catreg] per-seed Valid 0.69224 ± 0.00102 | stab 0.67585 ± 0.00077
[d_catreg] ensemble Valid 0.6961 | stab 0.6798 (mean 0.6929, a +0.00164, std_res 0.0262)
[d_catreg] so baseline (ensemble): ΔValid +0.0005 | Δstab -0.0002 | thắng 5/8 tuần | {44: 0.0016, 45: -0.0019, 46: -0.0031, 47: 0.0011, 48: -0.0006, 49: 0.0015, 50: 0.0028, 51: 0.0015}
set_mats: 620 feature | 0 categorical
[d_nocat] seed 42 | iter 515 | fit 0.8442 | Valid 0.6912 (Δ -0.0008) | gap 0.1530 | stab 0.6749 | 236s
[d_nocat] seed 43 | iter 439 | fit 0.8293 | Valid 0.6911 (Δ -0.0011) | gap 0.1382 | stab 0.6750 | 217s
[d_nocat] seed 44 | iter 370 | fit 0.8149 | Valid 0.6926 (Δ +

In [8]:
Y_FIT0 = y_fit
BASE_F = EXP['d_freq']['seeds']
PVAL_F = {s: np.load(REPORTS / f'lgb07_d_freq_pval_s{s}.npy') for s in SEEDS}

def freq_share(Xsrc):
    return {c: np.bincount(Xsrc[:, j][~np.isnan(Xsrc[:, j])].astype(np.int64),
                           minlength=len(CAT_MAPS[c])) / Xsrc.shape[0]
            for c, j in zip(HC, HC_IDX)}

def freq_cols_s(Xsrc, share):
    out = np.full((Xsrc.shape[0], len(HC)), np.nan, dtype=np.float32)
    for k, (c, j) in enumerate(zip(HC, HC_IDX)):
        v = Xsrc[:, j]; m = ~np.isnan(v)
        out[m, k] = share[c][v[m].astype(np.int64)]
    return out

def set_window(lo):
    """Cấu hình d_freq với cửa sổ fit tuần lo–39 (ES 40–43, Valid 44–51 giữ nguyên)."""
    global y_fit
    m = w_fit >= lo
    Xsrc = FIT0[m]
    share = freq_share(Xsrc)
    Xf = np.hstack([Xsrc[:, KEEP_D], freq_cols_s(Xsrc, share)])
    Xe = np.hstack([ES0[:, KEEP_D], freq_cols_s(ES0, share)])
    Xv = np.hstack([VAL0[:, KEEP_D], freq_cols_s(VAL0, share)])
    del Xsrc
    y_fit = Y_FIT0[m]
    set_mats(Xf, Xe, Xv, FEATS_D + FREQ_NAMES, CAT_D)
    print(f'Cửa sổ fit {lo}–39: {m.sum():,} dòng | dr {y_fit.mean():.4f}')

# ---- B: fit 20–39 ----
del Xf_f, Xe_f, Xv_f; gc.collect()
set_window(20)
EXP['f_win20'] = run_lgb('f_win20', base=BASE_F, base_pval=PVAL_F)

# ---- Khôi phục cấu hình tham chiếu d_freq (fit 0–39) ----
set_window(0)
assert len(y_fit) == len(Y_FIT0)

o = EXP['f_win20']
print(f"\nA (d_freq, 0–39): ensemble {EXP['d_freq']['ensemble']['gini_overall']:.4f} | "
      f"per-seed {EXP['d_freq']['gini_overall_mean_std'][0]:.4f}")
print(f"B (f_win20, 20–39): ensemble {o['ensemble']['gini_overall']:.4f} | "
      f"per-seed {o['gini_overall_mean_std'][0]:.4f} | gap {o['gap_fit_valid_mean_std'][0]:.4f} | "
      f"ΔValid {o['vs_base']['d_valid_ens']:+.4f} | thắng {o['vs_base']['weeks_won']}/8")

set_mats: 627 feature | 26 categorical
Cửa sổ fit 20–39: 436,016 dòng | dr 0.0293
[f_win20] seed 42 | iter 339 | fit 0.8665 | Valid 0.6879 (Δ -0.0053) | gap 0.1786 | stab 0.6717 | 140s
[f_win20] seed 43 | iter 317 | fit 0.8611 | Valid 0.6888 (Δ -0.0058) | gap 0.1723 | stab 0.6730 | 132s
[f_win20] seed 44 | iter 451 | fit 0.8934 | Valid 0.6892 (Δ -0.0038) | gap 0.2042 | stab 0.6730 | 162s

[f_win20] per-seed Valid 0.68863 ± 0.00065 | stab 0.67259 ± 0.00076
[f_win20] ensemble Valid 0.6932 | stab 0.6772 (mean 0.6899, a +0.00101, std_res 0.0254)
[f_win20] so baseline (ensemble): ΔValid -0.0045 | Δstab -0.0045 | thắng 0/8 tuần | {44: -0.0027, 45: -0.0017, 46: -0.0045, 47: -0.0057, 48: -0.0024, 49: -0.0077, 50: -0.0081, 51: -0.0029}
set_mats: 627 feature | 26 categorical
Cửa sổ fit 0–39: 754,796 dòng | dr 0.0288

A (d_freq, 0–39): ensemble 0.6977 | per-seed 0.6936
B (f_win20, 20–39): ensemble 0.6932 | per-seed 0.6886 | gap 0.1851 | ΔValid -0.0045 | thắng 0/8


In [9]:
from src.features import encode as E

lf = pl.scan_parquet(PROC / 'model_table.parquet')
STR_FEATS = [c for c in FEATS if schema[c] == pl.String]

# ---- Fit encoder trên tuần 0–39 (chỉ cần các cột String) ----
tr_str = lf.filter(pl.col('WEEK_NUM').is_between(*FIT_WEEKS)).select(STR_FEATS).collect(engine='streaming')
ENC = E.fit_encoder(tr_str, FEATS)
del tr_str
print(f"out_feats: {len(ENC['out_feats'])} | cat_cols: {len(ENC['cat_cols'])} | hc_cols: {len(ENC['hc_cols'])} | n_train: {ENC['n_train']:,}")

# ---- So khớp với ma trận d_freq đang có trong kernel ----
assert ENC['out_feats'] == FEATS_D + FREQ_NAMES
assert ENC['cat_cols'] == CAT_D
val_df = lf.filter(pl.col('WEEK_NUM').is_between(*VALID_WEEKS)).select(['case_id'] + FEATS).collect(engine='streaming')
assert np.array_equal(val_df['case_id'].to_numpy(), id_val), 'Thứ tự dòng Valid khác Cell 2'
Xv_mod = E.transform(val_df, ENC)
same = np.array_equal(Xv_mod, X_val, equal_nan=True)
print('Valid khớp d_freq:', same)
if not same:
    diff_cols = [ENC['out_feats'][j] for j in range(Xv_mod.shape[1])
                 if not np.array_equal(Xv_mod[:, j], X_val[:, j], equal_nan=True)]
    print('Cột lệch:', diff_cols[:20])
del val_df, Xv_mod; gc.collect()

# ---- Lưu encoder của cấu hình tham chiếu ----
E.save_encoder(ENC, REPORTS / 'enc_d_freq_w0_39.json')
print('Đã lưu reports/enc_d_freq_w0_39.json')

out_feats: 627 | cat_cols: 26 | hc_cols: 7 | n_train: 754,796
Valid khớp d_freq: True
Đã lưu reports/enc_d_freq_w0_39.json
